In [ ]:
from matplotlib.lines import Line2D
import sys

In [ ]:
helpers_folder = "./helpers"
sys.path.append(helpers_folder)

from plotting_functions import *
from data_processing import *

In [ ]:
with open("config/pathways_variable_list.yml", "r") as varfile:
    all_variables = yaml.safe_load(varfile)

VAR_MAPPING = generate_variable_mapping_from_list(all_variables, biomass_allocation=True)
VAR_MAPPING["sector - subsector"] = VAR_MAPPING[['sector', 'subsector']].agg(' - '.join, axis=1)
VAR_MAPPING["sector - fuel"] = VAR_MAPPING[['sector', 'fuel']].agg(' - '.join, axis=1)
var2sector = VAR_MAPPING.set_index("variable")["sector"].to_dict()
var2fuel = VAR_MAPPING.set_index("variable")["fuel"].to_dict()

In [ ]:
COLORMAPS = {}
for col in VAR_MAPPING:
    variables = list(VAR_MAPPING[col].unique())
    newdict = {}
    for sector, bcolor in basecolors.items():
        subset_variables = [v for v in variables if v.startswith(sector)]
        colors = gradient_from_base(bcolor, len(subset_variables))
        for v, c in zip(subset_variables, colors):
            newdict[v] = c
    COLORMAPS[col] = newdict
COLORMAPS["midpoint"] = midpoint2hex
COLORMAPS["midpoint (excl. CC)"] = midpoint2hex
COLORMAPS["sector"] = basecolors

In [ ]:
def get_default_label(fp):
    premise_run = fp.split("/")[-2]

    return premise_run.split("_")[0]

In [ ]:
output_folder = "output"
baseyear = 2025
expected_no_of_result_files = 53
use_adapted_pm = True
use_adapted_cc = True

## Data preparation

In [ ]:
labels = [
    "NPi",
    "1.5deg_base",
    "1.5deg_intervs",
]
data = get_plotting_data(output_folder, labels, VAR_MAPPING, resultsfolder="main", expected_no_of_result_files=expected_no_of_result_files,
                         use_adapted_pm=use_adapted_pm, use_adapted_cc=use_adapted_cc)
errdata = get_plotting_data(output_folder, labels, VAR_MAPPING, resultsfolder="MC", expected_no_of_result_files=expected_no_of_result_files,
                         datacols=["year", "impact_category", "sample index"],
                         use_adapted_pm=use_adapted_pm, use_adapted_cc=use_adapted_cc)
errdata_harmonized = harmonize_errordata(data, errdata)

In [ ]:
endpoints = ["ecosystem quality no LT", "ecosystem quality", "human health no LT", "human health", "natural resources", "natural resources no LT"]
mgroups = get_endpoint_method_groups(data[labels[0]], endpoints)

scen_year = [
    f"NPi - {baseyear}",
    "NPi - 2050",
    "1.5deg_base - 2050",
    "1.5deg_intervs - 2050",
]

columns_data = get_columnsdata(data, scen_year)
columns_errdata = get_columnsdata(errdata_harmonized, scen_year)

In [ ]:
def label_axes(axes, position=[-0.05, 1.1], fontsize=12, **kwargs):
    for i, ax in enumerate(axes):
        plbl = string.ascii_lowercase[i]
        ax.text(position[0], position[1], plbl, transform=ax.transAxes, fontsize=fontsize,
                fontweight='bold', va='top', ha='right', **kwargs)

## Natural resources plot

In [ ]:
ticklabels = [f"{str(baseyear)}", "CP", "NZ", "NZ-SCI"]
endpoints_plot = ["natural resources"]
p = 4
detail_midpoints = ["energy resources", "material resources"]
endpoint_scaling = {"natural resources": 1e-12}
ENDPOINT2UNIT.update({"natural resources": "trillion USD2013"})
MIDPOINT2UNIT.update(
    {mp: "trillion USD2013" for mp in detail_midpoints})
midpoint_scaling = {mp: 1e-12 for mp in detail_midpoints}

fig = plt.figure(layout="constrained", figsize=(10, 6))
subfigs = fig.subfigures(1, 2, wspace=0, width_ratios=[0.3, 0.7])

endpoint_stackplots(columns_data, subfigs[0], endpoints_plot,
                    mgroups, scen_year, ticklabels, errdata=columns_errdata,
                    scalings=endpoint_scaling, tickrotation=0)
midpoint_stackplots(columns_data, subfigs[1], detail_midpoints, 
                    scen_year, ticklabels, cols=["sector", "act_category"],errdata=columns_errdata, p=p, legend_y_offset=-0.01,
                    scalings=midpoint_scaling, tickrotation=0, fallback_unit="USD2013")
label_axes(fig.get_axes())
# fig.savefig("plot01_endpoint_overview_v2.pdf")


## Metal extraction plots

In [ ]:
metaldata = get_plotting_data(output_folder, labels, VAR_MAPPING, resultsfolder="metals",
                              datacols=["sector", "impact_category", "year"], expected_no_of_result_files=53,
                              use_adapted_pm=use_adapted_pm, use_adapted_cc=use_adapted_cc)
metaldataErr = get_plotting_data(output_folder, labels, VAR_MAPPING, resultsfolder="metalsMC",
                              datacols=["sample index", "impact_category", "year"], expected_no_of_result_files=53,
                              use_adapted_pm=use_adapted_pm, use_adapted_cc=use_adapted_cc)
metaldataErr_harmonized = harmonize_errordata(metaldata, metaldataErr)

scen_year = [
    f"NPi - {baseyear}",
    "NPi - 2050",
    "1.5deg_base - 2050",
    "1.5deg_intervs - 2050",
]

cdata_metals = get_columnsdata(metaldata, scen_year, add_midpoints=False, add_metals=True)
cdata_metalsErr = get_columnsdata(metaldataErr_harmonized, scen_year, add_midpoints=False, add_metals=True)
allmetals = cdata_metals["metal"].unique()

In [ ]:
refdata = pd.read_csv("../data/misc/commodities_production_2025.csv").set_index("Metal")["Value"]

In [ ]:
ticklabels = [f"{str(baseyear)}", "CP", "NZ", "NZ-SCI"]
ncols = 4
nrows = int(np.ceil(len(allmetals) / ncols))

fig, axs = plt.subplots(nrows, ncols, figsize=(12, 8), sharex=True)

for i, metal in enumerate(allmetals):
    ax = axs.flat[i]
    ax.set_title(metal)
    sel = cdata_metals[cdata_metals["metal"] == metal].groupby(
        ["scenario-year", "sector"]
    )["value"].sum().reset_index()
    selErr = cdata_metalsErr[cdata_metalsErr["metal"] == metal].pivot(
        index="scenario-year", columns="sample index", values="value").loc[scen_year]
    distributions = selErr.T.values * 1e-06
    pdata = sel.pivot(index="scenario-year", columns="sector", values="value").loc[scen_year] * 1e-06

    many_one_barplot(pdata, ax, basecolors, ticklabels, distributions=distributions)
    plbl = string.ascii_lowercase[i]
    refline = ax.axhline(refdata[metal], color="black", linestyle="--", lw=1, zorder=0)
    # ax.text(-0.05, 1.1, plbl, transform=ax.transAxes, fontsize=12,
                    # fontweight='bold', va='top', ha='right')

h, l = axs.flat[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper left", ncol=3, bbox_to_anchor=(0.1, -0.01))

fig.legend(handles=[Line2D([0], [0], color="black", linestyle="--", lw=1)],
           labels=["2025 global production\n(U.S. geological survey)"], title="Reference",
           loc="upper left", ncol=3, bbox_to_anchor=(0.6, -0.01))

fig.supylabel("Metal extraction [kt / yr]")
plt.tight_layout()

## All midpoints, by endpoint

In [ ]:
hh_extra_midpoints = ["climate change",  "ionising radiation", "ozone depletion", "photochemical oxidant formation", "water use"]
ic_filter = ["endpoint", "human health"]
ic_mask = ["total"]
p = 4
scalings = {mp: 1e-06 for mp in hh_extra_midpoints}

fig = plt.figure( figsize=(12, 13))
midpoint_stackplots(columns_data, fig, hh_extra_midpoints,
                    scen_year, ticklabels, errdata=columns_errdata, fallback_unit=None, p=p,
                    ic_filter=ic_filter, ic_mask=ic_mask, legend_y_offset=0.04, scalings=scalings,
                    legend_lw=30, tickrotation=0)
label_axes(fig.get_axes())
fig.supylabel("Human health impact [mil. DALY]")


In [ ]:
eq_extra_midpoints = ["acidification", "climate change",  "ecotoxicity",
                      "eutrophication", "photochemical oxidant formation", "water use"]
ic_filter = ["endpoint", "ecosystem quality"]
ic_mask = ["total"]
p = 4
scalings = {mp: 1e-03 for mp in eq_extra_midpoints}

fig = plt.figure( figsize=(12, 13))
midpoint_stackplots(columns_data, fig, eq_extra_midpoints,
                    scen_year, ticklabels, errdata=columns_errdata, fallback_unit=None, p=p,
                    ic_filter=ic_filter, ic_mask=ic_mask, legend_y_offset=0.04, scalings=scalings,
                    legend_lw=30, tickrotation=0)
label_axes(fig.get_axes())
fig.supylabel("Species loss [thousands]")